[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/4_DINO.ipynb)

**Día 2 · Cuaderno 1 de 3** · Acompaña al tema 4 de la web: [DINO y representaciones visuales](https://datamecum.sergiovillanuevalopez.com/temas/04-dino/)

**En clase:** las cinco partes. **Extra**, si da tiempo: parecidos y umbral de inspección. **Para casa:** la rejilla dibujada, un PCA común a varias fotos, el mapa de las piezas y tus propias piezas.

# DINO: describir imágenes sin etiquetas

DINO aprende a describir imágenes sin que nadie le diga qué hay en ellas. No clasifica ni detecta: convierte cada trozo de la imagen en un vector de números, y con esos vectores se pueden hacer muchas cosas.

Usamos **DINOv3** (Meta, agosto de 2025) y vemos, por orden:

1. Qué devuelve exactamente el modelo: el token CLS, los tokens de registro y un vector por parche.
2. Cuánto se parecen dos imágenes, con la similitud coseno.
3. Correspondencias entre parches de dos fotos distintas.
4. Los parches dibujados en color con PCA.
5. Inspección de piezas: decidir si una pieza es buena comparándola con ejemplos buenos y malos, y detectar anomalías cuando solo hay ejemplos buenos.

**Antes de empezar**

1. Activa la GPU: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.
2. Ejecuta las celdas en orden, de arriba abajo, con `Mayúsculas + Intro`.
3. No hay que instalar nada: todas las librerías que usamos ya vienen en Colab. Los modelos se descargan solos desde Hugging Face la primera vez que se usan.

Este cuaderno es tuyo: al abrirlo desde el enlace se crea una copia que solo ves tú, y puedes cambiar lo que quieras sin estropear el original.

## Preparación

In [ ]:
import torch
import timm
import requests
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from pathlib import Path
from PIL import Image
from sklearn.decomposition import PCA

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)

## El modelo

Los pesos oficiales de DINOv3 en la cuenta de Meta (`facebook/dinov3-...`) están restringidos: hay que pedir acceso y esperar a que lo aprueben. Para no depender de eso usamos la copia que publica `timm` en el Hub, la librería de modelos de visión de Hugging Face, que ya viene en Colab. Son los mismos pesos (timm avisa de pequeñas diferencias numéricas, que no se notan en este cuaderno).

La licencia es la de Meta (*DINOv3 License*): permite el uso comercial con condiciones. Si lo vas a usar en un producto, léela en la ficha del modelo.

Hay varios tamaños. Usamos el `base` (86 millones de parámetros), que da mapas más finos que el pequeño (`small`, 21 millones).

In [ ]:
MODELO = "timm/vit_base_patch16_dinov3.lvd1689m"
# MODELO = "timm/vit_small_patch16_dinov3.lvd1689m"   # más pequeño y rápido, menos detalle

# dynamic_img_size=True: acepta imágenes de cualquier tamaño (múltiplo del parche), no solo cuadradas
model = timm.create_model(f"hf-hub:{MODELO}", pretrained=True, dynamic_img_size=True).to(device).eval()

PARCHE = model.patch_embed.patch_size[0]
REGISTROS = model.num_prefix_tokens - 1          # tokens de registro: todos los prefijos menos el CLS
DIMENSION = model.embed_dim
MEDIA = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
DESVIACION = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

print(f"Modelo: {MODELO}")
print(f"  tamaño de parche: {PARCHE} x {PARCHE} píxeles")
print(f"  tokens de registro: {REGISTROS}")
print(f"  dimensión de cada vector: {DIMENSION}")

## Imágenes de ejemplo

In [ ]:
ASSETS = "https://raw.githubusercontent.com/sergiovillanueva/Modelos_Fundacionales/main/assets/"


def cargar_imagen(nombre_o_url):
    """Abre una imagen del curso en RGB.

    Si el cuaderno se ejecuta dentro de la carpeta del curso descargada (por ejemplo, sin internet),
    la toma de assets/; si no, la descarga del repositorio del curso o de la URL indicada.
    """
    local = Path("assets") / Path(nombre_o_url).name
    if local.exists():
        return Image.open(local).convert("RGB")
    url = nombre_o_url if nombre_o_url.startswith("http") else ASSETS + nombre_o_url
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content)).convert("RGB")


image_cars = cargar_imagen("cars.jpg")
image_person_dog = cargar_imagen("person_dog.jpg")
image_dog = cargar_imagen("dog.jpg")
image_dog2 = cargar_imagen("dog2.jpg")
image_fruits = cargar_imagen("fruits.jpg")
image_fruits3 = cargar_imagen("fruits3.jpg")
image_carpet_ok = cargar_imagen("carpet_ok.jpg")
image_carpet_nok = cargar_imagen("carpet_nok.jpg")
print("Imágenes cargadas")

## 1. Qué devuelve el modelo

DINO parte la imagen en cuadraditos de 16 × 16 píxeles (parches) y devuelve un vector por cada uno, más un vector `CLS` que resume la imagen entera. DINOv3 añade además cuatro **tokens de registro**: memoria interna del modelo que no corresponde a ninguna zona de la imagen.

Esto último importa: al separar los tokens hay que saltarse los registros. Si no, toda la rejilla queda desplazada cuatro posiciones y los mapas de color salen como ruido.

In [ ]:
def preparar(imagen, lado_largo=768):
    """Redimensiona a un múltiplo del tamaño de parche, sin deformar (si no, la rejilla no cuadra)."""
    ancho, alto = imagen.size
    escala = lado_largo / max(ancho, alto)
    nuevo_ancho = max(PARCHE, round(ancho * escala / PARCHE) * PARCHE)
    nuevo_alto = max(PARCHE, round(alto * escala / PARCHE) * PARCHE)
    return imagen.resize((nuevo_ancho, nuevo_alto), Image.BICUBIC)


def caracteristicas(imagen, lado_largo=768):
    """Pasa la imagen por DINO. Devuelve (cls, parches, rejilla, imagen_usada)."""
    lista = imagen if isinstance(imagen, (list, tuple)) else [imagen]
    preparadas = [preparar(img, lado_largo) for img in lista]

    salidas = []
    for img in preparadas:
        # Normalizar con la media y la desviación de ImageNet, que es como se entrenó el modelo
        x = torch.tensor(np.array(img) / 255.0, dtype=torch.float32).permute(2, 0, 1)
        x = ((x - MEDIA) / DESVIACION).unsqueeze(0).to(device)
        with torch.inference_mode():
            tokens = model.forward_features(x)[0]

        alto_px, ancho_px = x.shape[-2:]
        filas, columnas = alto_px // PARCHE, ancho_px // PARCHE

        cls = tokens[0].float().cpu().numpy()
        # Con DINOv3 hay que saltar el CLS y los tokens de registro
        parches = tokens[1 + REGISTROS:].float().cpu().numpy()

        if parches.shape[0] != filas * columnas:
            raise ValueError(
                f"Esperaba {filas * columnas} parches y hay {parches.shape[0]}. "
                "Revisa REGISTROS o el tamaño de entrada."
            )

        salidas.append((cls, parches, (filas, columnas), img))

    return salidas[0] if len(salidas) == 1 else salidas


# Veamos que sale exactamente
cls, parches, rejilla, usada = caracteristicas(image_dog)

print(f"Imagen original: {image_dog.size[0]} x {image_dog.size[1]} px")
print(f"Imagen usada:    {usada.size[0]} x {usada.size[1]} px")
print(f"Rejilla de parches: {rejilla[0]} filas x {rejilla[1]} columnas = {rejilla[0] * rejilla[1]} parches")
print(f"Tokens totales: 1 (CLS) + {REGISTROS} (registros) + {rejilla[0] * rejilla[1]} (parches)")
print(f"Cada parche es un vector de {parches.shape[1]} números")
print(f"La imagen entera cabe en {parches.shape[0] * parches.shape[1]:,} números")

## 2. Parecido entre imágenes: el token CLS

El vector `CLS` resume la imagen entera. Comparando dos con la similitud coseno sale un número entre -1 y 1. Con DINOv3 los números son bajos: fotos que no tienen nada que ver dan casi 0, y lo que importa es qué pareja destaca sobre las demás. Nadie le enseñó al modelo qué es un perro: lo que mide es si dos imágenes están descritas de forma parecida.

In [ ]:
def coseno(a, b):
    """Similitud coseno entre dos vectores."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def matriz_similitud(imagenes, nombres):
    """Compara todas las imágenes contra todas usando el vector CLS."""
    vectores = [caracteristicas(img)[0] for img in imagenes]
    n = len(vectores)
    matriz = np.array([[coseno(vectores[i], vectores[j]) for j in range(n)] for i in range(n)])

    figura, ejes = plt.subplots(1, 2, figsize=(15, 6),
                               gridspec_kw={"width_ratios": [1.1, 1]})

    ejes[0].imshow(np.hstack([np.array(img.resize((160, 160))) for img in imagenes]))
    ejes[0].set_title(" · ".join(nombres), fontsize=10)
    ejes[0].axis("off")

    mapa = ejes[1].imshow(matriz, cmap="viridis", vmin=matriz.min(), vmax=1)
    ejes[1].set_xticks(range(n), nombres, rotation=45, ha="right")
    ejes[1].set_yticks(range(n), nombres)
    for i in range(n):
        for j in range(n):
            ejes[1].text(j, i, f"{matriz[i, j]:.2f}", ha="center", va="center",
                         color="white" if matriz[i, j] < 0.8 else "black", fontsize=9)
    ejes[1].set_title("Similitud coseno entre imágenes (CLS)")
    plt.colorbar(mapa, ax=ejes[1], fraction=0.046)
    plt.tight_layout()
    plt.show()

    return matriz


_ = matriz_similitud(
    [image_dog, image_dog2, image_cars, image_fruits],
    ["perro", "perro 2", "coches", "frutas"],
)

## 3. Correspondencias entre parches

Elegimos **un parche** de una imagen y buscamos qué parche de la otra se le parece más. Si el modelo entiende lo que mira, el hocico de un perro encuentra el hocico del otro, aunque la raza, la postura y el fondo sean distintos.

Es la misma similitud coseno del cuaderno 3, aplicada región a región en lugar de a la imagen entera.

In [ ]:
def correspondencias(imagen_a, imagen_b, punto_relativo=(0.5, 0.4), lado_largo=512):
    """Elige un parche de imagen_a (punto_relativo de 0 a 1) y busca el más parecido en imagen_b."""
    (_, parches_a, (fa, ca), usada_a), (_, parches_b, (fb, cb), usada_b) = caracteristicas(
        [imagen_a, imagen_b], lado_largo
    )

    columna = min(int(punto_relativo[0] * ca), ca - 1)
    fila = min(int(punto_relativo[1] * fa), fa - 1)
    consulta = parches_a[fila * ca + columna]

    normal_b = parches_b / (np.linalg.norm(parches_b, axis=1, keepdims=True) + 1e-9)
    normal_q = consulta / (np.linalg.norm(consulta) + 1e-9)
    mapa = (normal_b @ normal_q).reshape(fb, cb)

    mejor = np.unravel_index(mapa.argmax(), mapa.shape)

    figura, ejes = plt.subplots(1, 2, figsize=(14, 5))

    ejes[0].imshow(usada_a)
    ejes[0].add_patch(plt.Rectangle((columna * PARCHE, fila * PARCHE), PARCHE, PARCHE,
                                    edgecolor="yellow", facecolor="none", linewidth=3))
    ejes[0].set_title("Parche elegido")
    ejes[0].axis("off")

    ejes[1].imshow(usada_b)
    ejes[1].imshow(mapa, cmap="inferno", alpha=0.6, extent=(0, usada_b.size[0], usada_b.size[1], 0))
    ejes[1].add_patch(plt.Rectangle((mejor[1] * PARCHE, mejor[0] * PARCHE), PARCHE, PARCHE,
                                    edgecolor="cyan", facecolor="none", linewidth=3))
    ejes[1].set_title(f"Dónde se parece más (max {mapa.max():.2f})")
    ejes[1].axis("off")

    plt.tight_layout()
    plt.show()

    return mapa


# La cabeza del perro de una foto, buscada en la otra
_ = correspondencias(image_dog, image_dog2, punto_relativo=(0.5, 0.35))

# Prueba también con:
# _ = correspondencias(image_person_dog, image_dog, punto_relativo=(0.35, 0.6))
# _ = correspondencias(image_fruits, image_fruits3, punto_relativo=(0.5, 0.5))

## 4. Ver los parches con PCA

Cada parche es un vector de cientos de números. Para verlo, se reduce a tres con PCA (análisis de componentes principales) y se pinta como rojo, verde y azul. Dos parches de colores parecidos son dos parches que el modelo describe de forma parecida.

Para que salga bien hay que cuidar dos cosas, y las dos están resueltas en las funciones de arriba:

1. **Saltarse los tokens de registro**, para que la rejilla no se descoloque.
2. **Usar un tamaño de imagen múltiplo del parche**, para que la rejilla cuadre con los tokens.

In [ ]:
def mapa_pca(parches, rejilla):
    """Convierte los parches en una imagen de colores con PCA."""
    filas, columnas = rejilla
    X = parches / np.linalg.norm(parches, axis=1, keepdims=True)   # solo cuenta la dirección
    componentes = PCA(n_components=3).fit_transform(X)
    # Llevar cada componente al rango 0-1 para usarla como color
    bajo = np.percentile(componentes, 2, axis=0)
    alto = np.percentile(componentes, 98, axis=0)
    componentes = np.clip((componentes - bajo) / (alto - bajo + 1e-9), 0, 1)
    return componentes.reshape(filas, columnas, 3)


def comparar_pca(imagen, lado_largo=896):
    """Enseña la imagen y sus parches en color."""
    _, parches, rejilla, usada = caracteristicas(imagen, lado_largo)
    figura, ejes = plt.subplots(1, 2, figsize=(14, 6))
    ejes[0].imshow(usada)
    ejes[0].set_title("Imagen")
    ejes[1].imshow(mapa_pca(parches, rejilla), interpolation="nearest")
    ejes[1].set_title("Parches en color (PCA)")
    for eje in ejes:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


comparar_pca(image_dog)
# comparar_pca(image_person_dog)

### Dónde mira el modelo: la atención

El PCA responde a "¿qué parches se parecen entre sí?", y por eso también colorea el césped o el cielo. La **atención** responde a otra pregunta: "¿en qué parches se fija el modelo para resumir la imagen?". Es la figura de la pantalla de atención de la web.

Para verla leemos los pesos de atención del vector CLS en la última capa del modelo. Nadie le dijo dónde estaba el perro.

In [ ]:
# Guardamos los pesos de atención de la última capa cada vez que pasa una imagen
ultima_capa = model.blocks[-1].attn
ultima_capa.fused_attn = False   # la versión rápida no deja ver los pesos
pesos = {}
ultima_capa.attn_drop.register_forward_hook(lambda capa, entrada, salida: pesos.update(atencion=salida))


def ver_atencion(imagen, lado_largo=768):
    """Pinta en qué parches se fija el CLS (media de todas las cabezas de atención)."""
    _, _, (filas, columnas), usada = caracteristicas(imagen, lado_largo)
    # Fila del CLS y columnas de los parches (sin el CLS ni los registros)
    atencion = pesos["atencion"][0, :, 0, 1 + REGISTROS:].mean(dim=0)
    atencion = atencion.float().cpu().numpy().reshape(filas, columnas)

    figura, ejes = plt.subplots(1, 2, figsize=(14, 6))
    ejes[0].imshow(usada)
    ejes[0].set_title("Imagen")
    ejes[1].imshow(atencion, cmap="inferno")
    ejes[1].set_title("Dónde mira el modelo (atención del CLS)")
    for eje in ejes:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


ver_atencion(image_dog)
ver_atencion(image_dog2)
# ver_atencion(image_person_dog)

## 5. Inspección de piezas: ¿buena o mala?

En una línea de producción la pregunta es siempre la misma: ¿esta pieza está bien? Con DINO hay dos caminos, según los ejemplos que tengamos:

- **Piezas buenas y piezas malas:** se compara la pieza nueva con las dos y se elige el grupo al que más se parece.
- **Solo piezas buenas:** se busca la zona que no se parece a nada de lo visto. Son las anomalías.

### Con ejemplos buenos y malos: ¿a cuál se parece?

Un caso típico de montaje: comprobar que un clip está puesto. Son recortes de la misma zona de un panel de puerta de coche. En las fotos OK el clip blanco está en su sitio; en las NOK falta y se ve el hueco vacío.

Damos como referencia dos fotos OK y dos NOK. Para cada foto nueva calculamos su parecido (coseno del CLS, como en la parte 2) con las OK y con las NOK, y gana el grupo con la media más alta. No hay entrenamiento: las cuatro fotos de referencia son todo el clasificador.

In [ ]:
image_clip_ok = [cargar_imagen("clip_ok_1.jpg"), cargar_imagen("clip_ok_2.jpg")]
image_clip_nok = [cargar_imagen("clip_nok_1.jpg"), cargar_imagen("clip_nok_2.jpg")]
image_clip_new = [cargar_imagen("clip_new_1.jpg"), cargar_imagen("clip_new_2.jpg")]


def a_cual_se_parece(buenas, malas, nuevas):
    """Dos referencias OK, dos NOK y dos fotos nuevas: decide OK o NOK con el CLS."""
    cls_buenas = [caracteristicas(img)[0] for img in buenas]
    cls_malas = [caracteristicas(img)[0] for img in malas]

    figura, ejes = plt.subplots(2, 4, figsize=(16, 8))
    for eje, img, titulo in zip(ejes[0], buenas + malas, ["Referencia OK"] * 2 + ["Referencia NOK"] * 2):
        eje.imshow(img)
        eje.set_title(titulo, color="green" if titulo.endswith(" OK") else "red", fontsize=13)

    for i, img in enumerate(nuevas):
        cls = caracteristicas(img)[0]
        parecido_ok = np.mean([coseno(cls, ref) for ref in cls_buenas])
        parecido_nok = np.mean([coseno(cls, ref) for ref in cls_malas])
        veredicto = "OK" if parecido_ok > parecido_nok else "NOK"

        ejes[1, 2 * i].imshow(img)
        ejes[1, 2 * i].set_title(f"Foto nueva {i + 1}: {veredicto}", fontsize=13,
                                 color="green" if veredicto == "OK" else "red")
        barras = ejes[1, 2 * i + 1]
        barras.barh(["se parece a NOK", "se parece a OK"], [parecido_nok, parecido_ok], color=["red", "green"])
        for fila, valor in enumerate([parecido_nok, parecido_ok]):
            barras.text(valor + 0.01, fila, f"{valor:.2f}", va="center", fontsize=12)
        barras.set_xlim(0, 1.15)
        barras.set_box_aspect(1)

    for eje in ejes.ravel():
        if eje.images:
            eje.axis("off")
    plt.tight_layout()
    plt.show()


a_cual_se_parece(image_clip_ok, image_clip_nok, image_clip_new)

### Solo con piezas buenas: anomalías

Lo de arriba funciona si tenemos fotos de piezas malas. Lo normal es lo contrario: miles de piezas buenas y casi ninguna mala, y cada mala falla de una forma distinta, así que no hay con qué comparar. Es el caso de la detección de anomalías, y la idea de **PatchCore**:

1. Pasar unas cuantas piezas **buenas** por DINO y guardar sus parches. Ese es el banco de normalidad, y es todo el entrenamiento que hay.
2. Para una pieza nueva, comparar cada parche con el parche más parecido del banco.
3. Si algún parche no se parece a nada de lo visto, ahí está el defecto.

Usamos una alfombra buena y otra con un nudo de hilos sueltos. El umbral se decide mirando solo la pieza buena: en la realidad no hay defectos de referencia.

**Un detalle técnico:** usamos una capa intermedia del modelo (`CAPA = 7`, el octavo de sus 12 bloques) y no la última. En las primeras pruebas, con la intermedia el defecto quedaba más localizado: cada parche describe sobre todo su propia zona, que es lo que queremos comparar. PatchCore hace algo parecido con su red.

**Ojo:** aquí la alfombra buena pasa por construcción, porque el umbral sale de ella misma. En una línea real el umbral se valida con otras piezas buenas que no estén en el banco.

In [ ]:
CAPA = 7   # el modelo tiene 12 capas: usamos una de la mitad (explicado arriba)


def parches_de_capa(imagen, capa=CAPA, lado_largo=640):
    """Vectores de los parches en una capa intermedia, ya normalizados."""
    usada = preparar(imagen, lado_largo)
    x = torch.tensor(np.array(usada) / 255.0, dtype=torch.float32).permute(2, 0, 1)
    x = ((x - MEDIA) / DESVIACION).unsqueeze(0).to(device)
    with torch.inference_mode():
        parches = model.forward_intermediates(x, indices=[capa], output_fmt="NLC", intermediates_only=True)[0][0]
    parches = parches.float().cpu().numpy()
    rejilla = (usada.size[1] // PARCHE, usada.size[0] // PARCHE)
    return parches / (np.linalg.norm(parches, axis=1, keepdims=True) + 1e-9), rejilla, usada


def banco_de_normalidad(imagenes_ok):
    """Guarda los parches de las piezas buenas. Es todo el entrenamiento que hay."""
    banco = np.concatenate([parches_de_capa(img)[0] for img in imagenes_ok])
    print(f"Banco con {banco.shape[0]} parches de {len(imagenes_ok)} imagen(es) buena(s)")
    return banco


def mapa_anomalia(banco, imagen, es_del_banco=False):
    """Distancia coseno de cada parche al parche bueno más parecido del banco.

    es_del_banco=True si la imagen está en el banco: cada parche se encontraría a sí mismo,
    así que se mira el segundo más parecido.
    """
    parches, (filas, columnas), usada = parches_de_capa(imagen)
    similitud = parches @ banco.T
    if es_del_banco:
        mejor = np.partition(similitud, -2, axis=1)[:, -2]
    else:
        mejor = similitud.max(axis=1)
    return (1.0 - mejor).reshape(filas, columnas), usada


def inspeccionar(banco, imagen, umbral, titulo="", es_del_banco=False):
    """Pinta el mapa de anomalía y decide si la pieza pasa."""
    mapa, usada = mapa_anomalia(banco, imagen, es_del_banco=es_del_banco)
    marcadas = int((mapa > umbral).sum())
    veredicto = "SE RECHAZA" if marcadas else "se acepta"

    figura, ejes = plt.subplots(1, 3, figsize=(16, 5))
    ejes[0].imshow(usada)
    ejes[0].set_title("Pieza")
    ejes[1].imshow(usada)
    # La misma escala de color en todas las piezas, para poder compararlas
    ejes[1].imshow(mapa, cmap="inferno", alpha=0.55, vmin=0, vmax=ESCALA,
                   extent=(0, usada.size[0], usada.size[1], 0))
    ejes[1].set_title(f"Mapa de anomalía (máx. {mapa.max():.3f})")
    ejes[2].imshow(usada)
    ejes[2].imshow((mapa > umbral).astype(float), cmap="Reds", alpha=0.5,
                   extent=(0, usada.size[0], usada.size[1], 0), interpolation="nearest")
    ejes[2].set_title(f"Por encima de {umbral:.3f}: {marcadas} parches")
    for eje in ejes:
        eje.axis("off")
    plt.suptitle(f"{titulo} → {veredicto}", fontsize=14)
    plt.tight_layout()
    plt.show()
    return mapa


banco = banco_de_normalidad([image_carpet_ok])

# El umbral se fija mirando solo la pieza buena: en producción no hay defectos de referencia.
# Dejamos un 20 % de margen por encima de lo peor que da una pieza correcta.
mapa_ok, _ = mapa_anomalia(banco, image_carpet_ok, es_del_banco=True)
umbral = float(mapa_ok.max()) * 1.2
print(f"Pieza buena: mediana {np.median(mapa_ok):.3f}, máxima {mapa_ok.max():.3f}")
print(f"Umbral elegido: {umbral:.3f}")
ESCALA = 2 * umbral   # la escala de color de los mapas, fija aunque cambies el umbral en el Extra

_ = inspeccionar(banco, image_carpet_ok, umbral, "Alfombra buena", es_del_banco=True)
_ = inspeccionar(banco, image_carpet_nok, umbral, "Alfombra con defecto")

## Extra: parecidos y umbral de inspección

Si vamos bien de tiempo lo hacemos juntos en clase; si no, queda para casa. Completa los huecos `___` y quita el `#` de las líneas de prueba.

In [ ]:
# Hueco 1: compara el perro con las dos fotos de frutas (image_fruits e image_fruits3).
# ¿Qué pareja sale más parecida?
# _ = matriz_similitud([image_dog, ___, ___], ["perro", "frutas", "frutas 2"])

# Hueco 2: multiplica el umbral por un número menor que 1. ¿Qué le pasa a la alfombra buena?
# _ = inspeccionar(banco, image_carpet_ok, umbral * ___, "Alfombra buena, umbral bajo", es_del_banco=True)

# Hueco 3: multiplícalo por un número mayor que 1 (prueba 2 y 3). ¿Cuándo se cuela el defecto?
# _ = inspeccionar(banco, image_carpet_nok, umbral * ___, "Alfombra con defecto, umbral alto")

## Para casa

Cuatro cosas más, ya comprobadas.

### Tus piezas y otro modelo

- **Tus piezas:** sube dos fotos con el icono de carpeta de la izquierda, una pieza buena y una con defecto, y monta el banco solo con la buena.
- **Otro modelo:** vuelve a la celda del modelo, usa `"timm/vit_small_patch16_dinov3.lvd1689m"`, más pequeño, y ejecuta otra vez. Compara los mapas de PCA y el máximo del mapa de anomalía.

In [ ]:
# mi_ok = Image.open("pieza_buena.jpg").convert("RGB")
# mi_nok = Image.open("pieza_mala.jpg").convert("RGB")
# mi_banco = banco_de_normalidad([mi_ok])
# El umbral se calcula otra vez, con tu pieza buena
# mi_mapa_ok, _ = mapa_anomalia(mi_banco, mi_ok, es_del_banco=True)
# mi_umbral = float(mi_mapa_ok.max()) * 1.2
# _ = inspeccionar(mi_banco, mi_nok, mi_umbral)

### La rejilla, dibujada

Cada celda es un parche y a cada una le corresponde un vector. Con parches de 16 px y una imagen de 768 px de lado largo salen unos 2.300 vectores por imagen.

In [ ]:
def ver_rejilla(imagen, lado_largo=768, cada=4):
    """Dibuja sobre la imagen la rejilla de parches que ve el modelo."""
    _, _, (filas, columnas), usada = caracteristicas(imagen, lado_largo)
    ancho, alto = usada.size

    plt.figure(figsize=(9, 9 * alto / ancho))
    plt.imshow(usada)
    for i in range(0, columnas + 1, cada):
        plt.axvline(i * PARCHE, color="white", linewidth=0.5, alpha=0.7)
    for j in range(0, filas + 1, cada):
        plt.axhline(j * PARCHE, color="white", linewidth=0.5, alpha=0.7)
    plt.title(f"{filas} x {columnas} parches de {PARCHE} px (una línea cada {cada})")
    plt.axis("off")
    plt.show()


ver_rejilla(image_dog)

### El mismo PCA para varias imágenes

Si el PCA se ajusta por separado en cada imagen, cada ajuste elige sus propias direcciones y el mismo color no significa lo mismo en dos imágenes. Ajustándolo una sola vez con los parches de todas, los colores pasan a ser comparables. Antes, a los vectores de cada imagen se les resta su media, para que el PCA compare cómo se diferencian los parches dentro de cada foto. Fíjate en que los tres perros salen del mismo color, también el que está tumbado en el sofá, aunque la raza, la postura y el fondo cambien por completo.

In [ ]:
def pca_compartido(imagenes, nombres, lado_largo=768):
    """Un solo PCA para varias imágenes: así el mismo color significa lo mismo en todas."""
    datos = caracteristicas(list(imagenes), lado_largo)
    # A cada imagen se le resta su media, para que el PCA compare el contenido y no el tono general de la foto
    centrados = []
    for _, parches, _, _ in datos:
        X = parches / np.linalg.norm(parches, axis=1, keepdims=True)
        centrados.append(X - X.mean(axis=0))
    pca = PCA(n_components=3).fit(np.concatenate(centrados))
    todos = pca.transform(np.concatenate(centrados))
    bajo, alto = np.percentile(todos, 2, axis=0), np.percentile(todos, 98, axis=0)

    figura, ejes = plt.subplots(2, len(datos), figsize=(5 * len(datos), 8))
    for columna, ((_, _, (filas, cols), usada), X) in enumerate(zip(datos, centrados)):
        color = np.clip((pca.transform(X) - bajo) / (alto - bajo + 1e-9), 0, 1)
        ejes[0, columna].imshow(usada)
        ejes[0, columna].set_title(nombres[columna])
        ejes[1, columna].imshow(color.reshape(filas, cols, 3), interpolation="nearest")
        for eje in ejes[:, columna]:
            eje.axis("off")
    plt.suptitle("Un mismo PCA para todas: el color significa lo mismo en las tres", y=0.98)
    plt.tight_layout()
    plt.show()


pca_compartido(
    [image_dog, image_dog2, image_person_dog],
    ["perro", "perro 2", "persona y perro"],
)

### El mismo resultado, en un mapa

Cada foto es un vector CLS de 768 números. Con PCA, como en la parte 4, lo bajamos a dos para poder dibujarlo: cada foto queda en un punto del plano, y dos fotos parecidas quedan cerca. Las OK forman un grupo, las NOK otro, y cada foto nueva cae junto al grupo al que se parece.

In [ ]:
from matplotlib.offsetbox import AnnotationBbox, OffsetImage


def mapa_de_parecidos(buenas, malas, nuevas):
    """Dibuja cada foto en el plano según su CLS (PCA a 2 dimensiones)."""
    imagenes = buenas + malas + nuevas
    colores = ["green"] * len(buenas) + ["red"] * len(malas) + ["royalblue"] * len(nuevas)
    vectores = np.array([caracteristicas(img)[0] for img in imagenes])
    vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
    puntos = PCA(n_components=2).fit_transform(vectores)

    figura, eje = plt.subplots(figsize=(9, 7))
    for (x, y), img, color in zip(puntos, imagenes, colores):
        miniatura = OffsetImage(np.array(img.resize((90, 90))), zoom=1)
        eje.add_artist(AnnotationBbox(miniatura, (x, y), bboxprops=dict(edgecolor=color, linewidth=4)))
    eje.scatter(puntos[:, 0], puntos[:, 1], alpha=0)   # para que los ejes se ajusten a los puntos
    eje.margins(0.3)
    eje.set_xticks([])
    eje.set_yticks([])
    eje.set_title("Verde: referencias OK · Rojo: referencias NOK · Azul: fotos nuevas")
    plt.show()


mapa_de_parecidos(image_clip_ok, image_clip_nok, image_clip_new)

## Resumen

- DINO devuelve tres tipos de token: CLS (la imagen entera), registros (memoria interna) y uno por parche. Hay que separarlos bien.
- Con el CLS y la similitud coseno se mide cuánto se parecen dos imágenes: importa qué pareja destaca, no el número suelto.
- Con los parches se encuentran correspondencias entre fotos distintas, y la atención dice dónde mira el modelo.
- Con PCA los parches se ven en color.
- Con unas pocas fotos de referencia buenas y malas, el CLS ya decide si una pieza es OK o NOK (por ejemplo, si un clip está puesto).
- Con un banco de parches buenos se detectan anomalías sin ningún ejemplo de defecto.

**La idea de fondo:** DINO no clasifica ni detecta, solo describe. Cada parche se convierte en un vector, y a partir de ahí todo son distancias. Por eso se le llama modelo fundacional: la misma descripción sirve para muchas tareas.

**Siguiente:** cuaderno 5, SAM 2. Antes, el tema 5 de la web. [Abrir el cuaderno en Colab](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/5_SAM.ipynb).